# Male CNS connectome (Janelia FlyEM, v1.0) -> input files for the death model

Runs on **Google Colab**, so the big files (~20 GB) are downloaded by Google, not by your own internet.
Everything ends up in your Google Drive in `malecns_v1/`. Only the small output files are needed on the laptop.

**What it makes** (same formats as the FlyWire files the model already reads, so the model needs almost no change):

| file | what | FlyWire equivalent |
|---|---|---|
| `synapse_cubes_male_10um.npz` | synapses per (neuron, 10 µm cube), outputs and inputs, **brain only** | `synapse_cubes_10um.npz` |
| `connections_male.csv` | pre, post, neuropil, synapse count (brain synapses only) | `connections_princeton.csv` |
| `neurons_male.csv` | neuron id + transmitter (ACH/GABA/GLUT/...) | `neurons.csv` |
| `coordinates_male.csv` | one point per neuron (centre of its brain synapses), nm | `coordinates.csv` |
| `classification_male.csv` | neuron id + superclass | `classification.csv` |
| `male_vs_female_check.txt` | numbers to compare with FlyWire before trusting a comparison | – |

**Brain only:** the male dataset also contains the ventral nerve cord (VNC). Synapses in VNC regions are left out,
so the result is comparable with FlyWire. Step 5 prints the region list it treats as VNC: **check it once**.

**Before you run:** Runtime → Change runtime type → if available pick **High-RAM**. The synapse files are read in pieces,
but a high-RAM runtime is safer. First run: ~20–40 min download, ~30–60 min processing.
Data: Janelia FlyEM Male CNS v1.0, CC-BY 4.0 (cite the Cell 2026 paper when using it).

In [ ]:
# 1. Google Drive + folders
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/malecns_v1'
RAW = f'{OUT}/raw'
os.makedirs(RAW, exist_ok=True)
KEEP_RAW_IN_DRIVE = True    # keep a copy of the big files in Drive, so later sessions skip the download
print('results ->', OUT)

In [ ]:
# 2. Download (resumable). Big files: syn-points 12.7 GB, syn-partners 6.8 GB
import shutil, subprocess
BASE = 'https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/'
FILES = ['body-annotations-male-cns-v1.0-minconf-0.5.feather',
         'body-neurotransmitters-male-cns-v1.0.feather',
         'syn-points-male-cns-v1.0-minconf-0.5.feather',
         'syn-partners-male-cns-v1.0-minconf-0.5.feather']
for name in FILES:
    local, saved = f'/content/{name}', f'{RAW}/{name}'
    if os.path.exists(local):
        print(name, 'already on Colab disk')
    elif os.path.exists(saved):
        print(name, 'copying from Drive ...')
        shutil.copy(saved, local + '.part'); os.replace(local + '.part', local)
    else:
        print(name, 'downloading ...')
        subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', local + '.part', BASE + name], check=True)
        os.replace(local + '.part', local)
    if KEEP_RAW_IN_DRIVE and not os.path.exists(saved):
        print('   copy to Drive ...')
        shutil.copy(local, saved + '.part'); os.replace(saved + '.part', saved)
    print(f'   {name}: {os.path.getsize(local)/1e9:.2f} GB')

In [ ]:
# 3. Look at what is inside (column names are detected from this, not assumed)
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.ipc as ipc, time
def open_feather(name):
    return ipc.open_file(pa.memory_map(f'/content/{name}', 'r'))
for name in FILES:
    f = open_feather(name)
    print(f'\n=== {name}: {f.num_record_batches} record batches')
    print(f.schema)

import pyarrow.compute as pc
def strcol(batch, name):
    '''column as plain strings (dictionary-encoded columns are decoded)'''
    col = batch.column(name)
    if pa.types.is_dictionary(col.type):
        col = col.dictionary_decode()
    return col.cast(pa.string())
def num(batch, name):
    return batch.column(name).to_numpy(zero_copy_only=False)

def pick(columns, *candidates, required=True):
    '''first candidate column name that exists (case-insensitive)'''
    low = {c.lower(): c for c in columns}
    for c in candidates:
        if c.lower() in low:
            return low[c.lower()]
    if required:
        raise KeyError(f'none of {candidates} in {list(columns)}')
    return None

In [ ]:
# 4. Neurons: which bodies count, their transmitter and superclass
ann = pd.read_feather('/content/' + FILES[0])
nt = pd.read_feather('/content/' + FILES[1])
print(ann.columns.tolist()); print(nt.columns.tolist())
A_ID = pick(ann.columns, 'bodyId', 'body', 'id')
A_STATUS = pick(ann.columns, 'status', required=False)
A_SUPER = pick(ann.columns, 'superclass', 'superClass', 'super_class', required=False)
N_ID = pick(nt.columns, 'bodyId', 'body', 'id')
N_NT = pick(nt.columns, 'consensus_nt', 'predicted_nt', 'celltype_predicted_nt', 'nt', 'top_nt')
if A_STATUS:
    print(ann[A_STATUS].value_counts(dropna=False))
# Same idea as FlyWire 'proofread': keep traced bodies. Check the printed statuses and edit if needed.
GOOD_STATUS = {'Traced', 'Anchor', 'Roughly traced', 'Prelim Roughly traced', 'Sensory Anchor', 'Cervical Anchor', 'Soma Anchor', 'Primary Anchor'}
keep = ann[ann[A_STATUS].isin(GOOD_STATUS)] if A_STATUS else ann
neurons = keep[[A_ID] + ([A_SUPER] if A_SUPER else [])].rename(columns={A_ID: 'root_id', A_SUPER: 'super_class'} if A_SUPER else {A_ID: 'root_id'})
neurons = neurons.merge(nt[[N_ID, N_NT]].rename(columns={N_ID: 'root_id', N_NT: 'nt_raw'}), on='root_id', how='left')
NT_MAP = {'acetylcholine': 'ACH', 'ach': 'ACH', 'gaba': 'GABA', 'glutamate': 'GLUT', 'glut': 'GLUT',
          'dopamine': 'DA', 'da': 'DA', 'serotonin': 'SER', 'ser': 'SER', '5ht': 'SER',
          'octopamine': 'OCT', 'oct': 'OCT', 'histamine': 'HIST', 'his': 'HIST', 'tyramine': 'TYR'}
neurons['nt_type'] = neurons['nt_raw'].astype(str).str.lower().map(NT_MAP)
print(neurons['nt_raw'].value_counts(dropna=False).head(15))
print(f'{len(neurons):,} traced bodies; transmitter shares:')
print(neurons['nt_type'].value_counts(normalize=True, dropna=False).round(3))
print('(the model treats HIST/TYR/unknown as sign 0, like DA/SER/OCT in FlyWire)')
ids = np.sort(neurons['root_id'].values.astype(np.int64))

In [ ]:
# 5. Pass 1 over synapse points: region names and grid extent
pts = open_feather(FILES[2])
cols = pts.schema.names
P_BODY = pick(cols, 'body', 'bodyId', 'body_id')
P_KIND = pick(cols, 'kind', 'type')
P_ROI = pick(cols, 'roi', 'primary_roi', 'neuropil')
P_X, P_Y, P_Z = pick(cols, 'x'), pick(cols, 'y'), pick(cols, 'z')
VOXEL_NM = 8.0                       # coordinates are in 8 nm voxels
roi_count = {}
lo = np.full(3, np.inf); hi = np.full(3, -np.inf)
t0 = time.time()
for b in range(pts.num_record_batches):
    batch = pts.get_batch(b)
    for item in pc.value_counts(strcol(batch, P_ROI).fill_null('None')).to_pylist():
        roi_count[item['values']] = roi_count.get(item['values'], 0) + item['counts']
    print(f'batch {b+1}/{pts.num_record_batches} ({time.time()-t0:.0f} s)', end='\r')
roi_count = pd.Series(roi_count).sort_values(ascending=False)
# VNC regions in the male CNS: leg / wing / haltere / neck neuropils, tectulum, abdominal and ovoid neuropils
VNC_KEYS = ['LegNp', 'Tct', 'ANm', 'Ov', 'mVAC', 'AbN', 'VNC', 'NTct', 'WTct', 'HTct', 'LTct', 'IntTct', '(T1)', '(T2)', '(T3)', 'T1)', 'T2)', 'T3)']
is_vnc = pd.Series({r: any(k in r for k in VNC_KEYS) for r in roi_count.index})
NONE_NAMES = {'None', 'nan', '<NA>', ''}
print(f'\n{len(roi_count)} region labels. Treated as VNC (check this list!):')
print(roi_count[is_vnc].to_string())
print('\nBrain regions (top 40):'); print(roi_count[~is_vnc & ~roi_count.index.isin(NONE_NAMES)].head(40).to_string())
print(f'points without region: {roi_count[roi_count.index.isin(NONE_NAMES)].sum():,} of {roi_count.sum():,} (left out)')
BRAIN_ROIS = set(roi_count.index[~is_vnc & ~roi_count.index.isin(NONE_NAMES)])

In [ ]:
# 6. Pass 2: grid over BRAIN synapses, then count synapses per (neuron, cube) for outputs and inputs
CUBE_UM = 10
S = CUBE_UM * 1000.0
roi_names = sorted(BRAIN_ROIS)

ROI_SET = pa.array(roi_names)
def brain_xyz(batch):
    '''brain points of a batch: mask, coordinates in nm, region index'''
    r = pc.index_in(strcol(batch, P_ROI), value_set=ROI_SET).fill_null(-1).to_numpy(zero_copy_only=False)
    ok = r >= 0
    xyz = np.stack([num(batch, c) for c in (P_X, P_Y, P_Z)], 1)[ok].astype(np.float64) * VOXEL_NM
    return ok, xyz, r[ok]

for b in range(pts.num_record_batches):            # extent of brain synapses
    ok, xyz, _ = brain_xyz(pts.get_batch(b))
    if len(xyz):
        lo = np.minimum(lo, xyz.min(0)); hi = np.maximum(hi, xyz.max(0))
lo = np.floor(lo).astype(np.int64); hi = np.ceil(hi).astype(np.int64)
dims = ((hi - lo) // S + 1).astype(np.int64); n_cubes = int(np.prod(dims))
print('brain extent (µm):', (hi - lo) / 1000, ' grid:', dims, f'= {n_cubes:,} cubes')

def to_idx(body):
    k = np.clip(np.searchsorted(ids, body), 0, len(ids) - 1)
    return np.where(ids[k] == body, k, -1)
def reduce(keys, counts):
    u, inv = np.unique(keys, return_inverse=True)
    return u, np.bincount(inv, weights=counts).astype(np.int64)

acc = {'pre': [], 'post': []}
NR = len(roi_names)
cube_roi = np.zeros(n_cubes * NR, np.int64)                    # synapse points per (cube, region)
sum_xyz = np.zeros((len(ids), 3)); n_pts = np.zeros(len(ids))
t0 = time.time()
for b in range(pts.num_record_batches):
    batch = pts.get_batch(b)
    ok, xyz, roi = brain_xyz(batch)
    body = to_idx(num(batch, P_BODY).astype(np.int64)[ok])
    kind = pc.utf8_lower(strcol(batch, P_KIND))
    is_pre = pc.match_substring(kind, 'pre').fill_null(False).to_numpy(zero_copy_only=False)[ok]
    is_post = pc.match_substring(kind, 'post').fill_null(False).to_numpy(zero_copy_only=False)[ok]
    c = ((xyz - lo) // S).astype(np.int64)
    cube = c[:, 0] + dims[0] * (c[:, 1] + dims[1] * c[:, 2])
    cube_roi += np.bincount(cube * NR + roi, minlength=n_cubes * NR)
    good = body >= 0
    for a in range(3):
        sum_xyz[:, a] += np.bincount(body[good], weights=xyz[good, a], minlength=len(ids))
    n_pts += np.bincount(body[good], minlength=len(ids))
    for side, flag in [('pre', is_pre), ('post', is_post)]:
        m = good & flag
        u, cnt = np.unique(body[m] * n_cubes + cube[m], return_counts=True)
        acc[side].append((u, cnt))
        if len(acc[side]) >= 8:
            k = np.concatenate([a for a, _ in acc[side]]); w = np.concatenate([x for _, x in acc[side]])
            acc[side] = [reduce(k, w)]
    print(f'batch {b+1}/{pts.num_record_batches} ({time.time()-t0:.0f} s)', end='\r')
result = {}
for side in ['pre', 'post']:
    k = np.concatenate([a for a, _ in acc[side]]); w = np.concatenate([x for _, x in acc[side]])
    u, cnt = reduce(k, w)
    result[side] = (u // n_cubes, u % n_cubes, cnt)
    print(f'\n{side}: {len(u):,} (neuron, cube) pairs, {cnt.sum():,} brain synapses of traced neurons')

In [ ]:
# 7. Save the cube file (same layout as the FlyWire one; the model reads it unchanged)
in_brain = n_pts > 0
print(f'{in_brain.sum():,} of {len(ids):,} traced bodies have brain synapses (the rest are VNC-only and are dropped)')
keep_ids = ids[in_brain]
remap = np.full(len(ids), -1, np.int64); remap[in_brain] = np.arange(in_brain.sum())
out = {}
for side in ['pre', 'post']:
    n, c, cnt = result[side]
    n = remap[n]; ok = n >= 0
    out[side] = (n[ok], c[ok], cnt[ok])
fname = f'{OUT}/synapse_cubes_male_{CUBE_UM}um.npz'
np.savez_compressed(fname, root_ids=keep_ids, cube_um=CUBE_UM, origin_nm=lo, dims=dims, position='point',
    pre_neuron=out['pre'][0].astype(np.int32), pre_cube=out['pre'][1].astype(np.int32), pre_count=out['pre'][2].astype(np.int32),
    post_neuron=out['post'][0].astype(np.int32), post_cube=out['post'][1].astype(np.int32), post_count=out['post'][2].astype(np.int32))
print('saved', fname, f'({os.path.getsize(fname)/1e6:.0f} MB)')
cube_roi = cube_roi.reshape(n_cubes, NR)
cube_main_roi = np.where(cube_roi.sum(1) > 0, cube_roi.argmax(1), -1)    # region label per cube

In [ ]:
# 8. Connections between kept neurons, counted from synapse partners whose post point lies in a brain cube
par = open_feather(FILES[3])
cols = par.schema.names
Q_PRE, Q_POST = pick(cols, 'body_pre', 'pre_body', 'bodyId_pre'), pick(cols, 'body_post', 'post_body', 'bodyId_post')
Q_X, Q_Y, Q_Z = pick(cols, 'x_post'), pick(cols, 'y_post'), pick(cols, 'z_post')
NK = len(keep_ids)
def to_keep(body):
    k = np.clip(np.searchsorted(keep_ids, body), 0, NK - 1)
    return np.where(keep_ids[k] == body, k, -1)
accp = []; t0 = time.time(); n_all = n_brain = 0
for b in range(par.num_record_batches):
    batch = par.get_batch(b)
    pre = to_keep(num(batch, Q_PRE).astype(np.int64))
    post = to_keep(num(batch, Q_POST).astype(np.int64))
    xyz = np.stack([num(batch, c) for c in (Q_X, Q_Y, Q_Z)], 1).astype(np.float64) * VOXEL_NM
    c = ((xyz - lo) // S).astype(np.int64)
    inside = np.all((c >= 0) & (c < dims), 1)
    cube = np.where(inside, c[:, 0] + dims[0] * (c[:, 1] + dims[1] * c[:, 2]), 0)
    roi = np.where(inside, cube_main_roi[cube], -1)
    ok = (pre >= 0) & (post >= 0) & (roi >= 0)
    n_all += len(pre); n_brain += ok.sum()
    key = (pre[ok] * NK + post[ok]) * NR + roi[ok]
    u, cnt = np.unique(key, return_counts=True); accp.append((u, cnt))
    if len(accp) >= 8:
        k = np.concatenate([a for a, _ in accp]); w = np.concatenate([x for _, x in accp]); accp = [reduce(k, w)]
    print(f'batch {b+1}/{par.num_record_batches} ({time.time()-t0:.0f} s)', end='\r')
k = np.concatenate([a for a, _ in accp]); w = np.concatenate([x for _, x in accp]); u, cnt = reduce(k, w)
conn = pd.DataFrame({'pre_root_id': keep_ids[u // NR // NK], 'post_root_id': keep_ids[(u // NR) % NK],
                     'neuropil': np.array(roi_names)[u % NR], 'syn_count': cnt})
print(f'\n{n_brain:,} of {n_all:,} synapse pairs are between kept neurons inside the brain')
# FlyWire connection tables leave out weak pairs. Same rule here: pairs with fewer than SYN_MIN synapses in total are dropped.
SYN_MIN = 5          # check against the FlyWire file (smallest pair total there) before comparing sexes
tot = conn.groupby(['pre_root_id', 'post_root_id'])['syn_count'].transform('sum')
conn = conn[tot >= SYN_MIN]
conn.to_csv(f'{OUT}/connections_male.csv', index=False)
print(f'saved connections_male.csv: {len(conn):,} rows, {conn.groupby(["pre_root_id","post_root_id"]).ngroups:,} pairs')

In [ ]:
# 9. Neuron files in FlyWire layout
kept = neurons.set_index('root_id').loc[keep_ids].reset_index()
kept[['root_id', 'nt_type']].to_csv(f'{OUT}/neurons_male.csv', index=False)
cent = sum_xyz[in_brain] / n_pts[in_brain, None]
pd.DataFrame({'root_id': keep_ids,
              'position': [f'[{x:.0f} {y:.0f} {z:.0f}]' for x, y, z in cent]}).to_csv(f'{OUT}/coordinates_male.csv', index=False)
(kept[['root_id', 'super_class']] if 'super_class' in kept else kept[['root_id']].assign(super_class='unknown')
 ).to_csv(f'{OUT}/classification_male.csv', index=False)
print('saved neurons_male.csv, coordinates_male.csv, classification_male.csv')

In [ ]:
# 10. Numbers to compare with FlyWire before any male-vs-female comparison
occupied = np.unique(np.concatenate([out['pre'][1], out['post'][1]]))
pair = conn.groupby(['pre_root_id', 'post_root_id'])['syn_count'].sum()
lines = [
    f'brain neurons kept: {len(keep_ids):,}   (FlyWire female: 139,255)',
    f'connections (pairs >= {SYN_MIN} synapses): {len(pair):,}   (FlyWire: 3,732,460 pair-neuropil rows)',
    f'synapses per pair: median {pair.median():.0f}, mean {pair.mean():.1f}, 90th pct {pair.quantile(.9):.0f}',
    f'brain synapses: outputs {out["pre"][2].sum():,}, inputs {out["post"][2].sum():,}',
    f'neuropil cubes with synapses: {len(occupied):,}   (FlyWire: 23,045 used by the model)',
    'transmitters: ' + ', '.join(f'{k} {v:.1%}' for k, v in kept['nt_type'].value_counts(normalize=True, dropna=False).items()),
    'NOTE: male and FlyWire differ in sex, individual, AND reconstruction pipeline (synapse detection, proofreading).',
    '      Compare synapse-count scales before using the same W_SYN; differences are not automatically sex differences.',
]
txt = '\n'.join(lines); print(txt)
open(f'{OUT}/male_vs_female_check.txt', 'w').write(txt + '\n')
print('\nDone. Small files in', OUT, ':', [f for f in os.listdir(OUT) if not f.startswith('raw')])